# Hephaestus — CVE Analysis Automaton (LoRA, Qwen2.5-3B)
Model: Qwen/Qwen2.5-3B-Instruct  |  Method: QLoRA + SFT (unsloth)  |  GPU: P100 (sm_60, cu117)

Closes Hephaestus v0.2 Plan 1.5. Mirrors the proven Heimdall notebook (unsloth + torch 2.0.1+cu117),
not the harness's `setup.py` (which installs untested torch 2.5.1 and whose SFTConfig is contradictory for this task).

IMPORTANT: the CVE dataset `AlicanKiraz0/All-CVE-Training-Dataset` is **gated**. Add your HF token to a Kaggle secret
named `HF_TOKEN` (or set it in the first code cell). If the schema differs from {cve_id, description, severity} the guard cell will FAIL LOUD rather than train on zero rows.

In [ ]:
# --- 0. Secrets / env ---
# Set HF_TOKEN here OR via Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN).
import os
from kaggle_secrets import UserSecretsClient

try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secret')
except Exception as e:
    print(f'No Kaggle secret HF_TOKEN ({e}); using inline env if set. Paste your token below if needed.')
    # os.environ['HF_TOKEN'] = 'hf_xxx'  # uncomment to inline (do NOT commit)

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# --- 1. Install torch 2.0.1 + cu117 (P100 sm_60 compatible) + unsloth stack ---
# Proven in Heimdall run. Overrides Kaggle default torch (cu128, sm_70+) which crashes on P100.
%pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu117
%pip install -q unsloth transformers==4.46.3 datasets trl==0.8.6 peft==0.13.2 accelerate bitsandbytes==0.46.1

In [ ]:
# --- 2. Load + inspect the GATED CVE dataset (with schema guard) ---
import pandas as pd
from datasets import load_dataset

DS = 'AlicanKiraz0/All-CVE-Training-Dataset'
ds = load_dataset(DS, token=os.environ.get('HF_TOKEN'))
print('Splits:', list(ds.keys()))
for split in ds:
    print(f'  {split}: {len(ds[split])} rows')

# Inspect schema of the first row
sample = ds['train'][0]
print('Columns:', list(sample.keys()))
print('Sample:', {k: str(v)[:120] for k, v in sample.items()})

# GUARD: we expect cve_id / description / severity. If absent, STOP and fix the converter.
expected = {'description', 'severity'}
have = set(sample.keys())
missing = expected - have
assert not missing, f'SCHEMA MISMATCH: missing {missing}. Got {have}. Fix _convert_to_messages below.'
print('Schema OK:', sorted(have))

In [ ]:
# --- 3. Convert to messages format (mirrors hephaestus/loader.py CVE branch) ---
# If the guard above flagged different column names, edit the field reads here.
import json, re

CLASSES = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
SYSTEM = ('You are a cybersecurity expert specializing in CVE analysis. '
          'Given a CVE description, classify its severity as CRITICAL, HIGH, MEDIUM, or LOW, '
          'and provide a brief risk assessment.')

def to_messages(item):
    cve_id = item.get('cve_id', 'Unknown')
    description = item.get('description', item.get('text', ''))
    severity = str(item.get('severity', item.get('label', ''))).strip().upper()
    # Normalise common variants to the 4 canonical classes
    if severity not in CLASSES:
        sev_map = {'CRIT': 'CRITICAL', 'C': 'CRITICAL', 'H': 'HIGH', 'M': 'MEDIUM', 'L': 'LOW'}
        severity = sev_map.get(severity[:1], 'MEDIUM')
    user = f'Analyze this CVE:\n\nCVE ID: {cve_id}\nDescription: {description}'
    return [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': f'{severity}. ' + ('Requires immediate attention.' if severity in ('CRITICAL','HIGH') else 'Monitor and patch as routine.')},
    ]

MAX_TRAIN = 50000
raw_train = ds['train'].select(range(min(MAX_TRAIN, len(ds['train']))))
raw_test = ds['test'] if 'test' in ds else ds['train'].select(range(len(ds['train']), len(ds['train'])))  # fallback; see note

train_msgs = [{'messages': to_messages(r)} for r in raw_train]
# Hold out last 1000 of train as a clean test if no 'test' split exists
if 'test' not in ds:
    train_msgs, test_msgs = train_msgs[:-1000], train_msgs[-1000:]
else:
    test_msgs = [{'messages': to_messages(r)} for r in raw_test.select(range(min(1000, len(raw_test))))]

assert train_msgs and test_msgs, 'Produced empty train/test — converter or split wrong.'
print(f'Converted: train={len(train_msgs)} test={len(test_msgs)}')
print('Example:', json.dumps(train_msgs[0], indent=2)[:400])

In [ ]:
# --- 4. Build HF dataset from messages ---
from datasets import Dataset
train_ds = Dataset.from_list(train_msgs)
test_ds = Dataset.from_list(test_msgs)
print('train', len(train_ds), 'test', len(test_ds))

In [ ]:
# --- 5. Load Qwen2.5-3B with unsloth 4-bit ---
from unsloth import FastLanguageModel
import torch

MODEL = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    dtype=torch.float16,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=256, lora_alpha=512, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
print('LoRA applied')

In [ ]:
# --- 6. Train ---
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=test_ds,
    dataset_text_field='messages', max_seq_length=MAX_SEQ,
    args=TrainingArguments(
        per_device_train_batch_size=1, per_device_eval_batch_size=1,
        gradient_accumulation_steps=2, warmup_steps=20, max_steps=200,
        learning_rate=2e-5, fp16=True, logging_steps=10,
        eval_strategy='steps', eval_steps=40, save_strategy='steps', save_steps=40,
        load_best_model_at_end=True, metric_for_best_model='eval_loss',
        output_dir='/kaggle/working/outputs', report_to='none',
    ),
)
trainer.train()

In [ ]:
# --- 7. Evaluate (multiclass, mirrors hephaestus/evaluator.py _evaluate_multiclass) ---
import re
from collections import defaultdict
model.eval()

def extract_label(text, classes):
    t = text.upper().strip()
    for c in sorted(classes, key=len, reverse=True):
        if re.search(r'\b' + re.escape(c) + r'\b', t):
            return c
    return classes[0]

per_tp, per_fp, per_fn, per_corr, per_tot = (defaultdict(int) for _ in range(5))
for item in test_msgs:
    msgs = item['messages']
    expected = next(m['content'] for m in reversed(msgs) if m['role']=='assistant')
    exp = extract_label(expected, CLASSES); per_tot[exp]+=1
    prompt = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    inp = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=64, do_sample=False)
    resp = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
    pred = extract_label(resp, CLASSES)
    if pred==exp: per_corr[exp]+=1; per_tp[pred]+=1
    else: per_fn[exp]+=1; per_fp[pred]+=1

n=len(test_msgs); correct=sum(per_corr.values())
acc=correct/n
ps=[per_tp[c]/(per_tp[c]+per_fp[c]) if per_tp[c]+per_fp[c] else 0 for c in CLASSES]
rs=[per_tp[c]/(per_tp[c]+per_fn[c]) if per_tp[c]+per_fn[c] else 0 for c in CLASSES]
fs=[2*p*r/(p+r) if p+r else 0 for p,r in zip(ps,rs)]
print(f'Accuracy: {acc*100:.1f}% | Macro-F1: {sum(fs)/4*100:.1f}%')
for c in CLASSES:
    a = per_corr[c]/per_tot[c] if per_tot[c] else 0
    print(f'  {c}: acc={a*100:.1f}% (n={per_tot[c]})')
GATE=0.95
print(f'\nQuality Gate (95%): {"PASS" if acc>=GATE else "FAIL"}')

In [ ]:
# --- 8. Merge, save, and push to HuggingFace as Yusif-v/hephaestus-cve-analyzer ---
model.push_to_hub_merged(
    'Yusif-v/hephaestus-cve-analyzer',
    tokenizer=tokenizer,
    token=os.environ.get('HF_TOKEN'),
    save_method='merged_16bit',
)
print('Pushed Yusif-v/hephaestus-cve-analyzer')